# M3. 기술통계 · 분석 (03_m3_기술통계분석)

`data/processed`(M2 결과)로 사고사례와 업종·규모 통계를 분석합니다.

| 체크리스트 | 이 노트북의 위치 |
|---|---|
| M3-1 기술통계 요약 | 1절 |
| M3-2 분포·추이 분석 | 2절 |
| M3-3 주요 범주 비교 | 3절 |
| M3-4 변수 간 관계 분석 | 4절 |
| M3-5 시각화 작성 | 2~4절 그래프 5개 |
| M3-6 TF-IDF·통계검정 (필요 시) | 4절 상관검정만 수행, TF-IDF는 생략 |
| M3-7 핵심 인사이트 정리 | 5절 |

위에서부터 순서대로 실행하세요. 분석 표는 `data/analysis/m3_*.csv`로 저장됩니다.

## 0. 준비

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import font_manager

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)

# 한글 글꼴 (맥: AppleGothic, 윈도우: Malgun Gothic)
installed = {f.name for f in font_manager.fontManager.ttflist}
for name in ["AppleGothic", "Malgun Gothic", "NanumGothic", "Noto Sans CJK KR"]:
    if name in installed:
        plt.rcParams["font.family"] = name
        break
plt.rcParams["axes.unicode_minus"] = False

# data/processed가 있는 폴더를 위로 올라가며 찾는다
ROOT = Path.cwd()
while not (ROOT / "data" / "processed").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
P = ROOT / "data" / "processed"
A = ROOT / "data" / "analysis"
A.mkdir(parents=True, exist_ok=True)

def rd(prefix):
    """processed 폴더에서 prefix로 시작하는 CSV를 읽는다."""
    hits = sorted(P.glob(f"{prefix}*.csv"))
    assert hits, f"'{prefix}'로 시작하는 processed 파일이 없습니다. 02 노트북을 먼저 실행하세요."
    return pd.read_csv(hits[0], encoding="utf-8-sig")

mf, cs = rd("1_SIF_제조업등"), rd("1_SIF_건설업")
sif = pd.concat([mf, cs], ignore_index=True)
inj, dead, firms, rate = rd("2_"), rd("3_"), rd("4_"), rd("5_")
print("사고사례:", len(sif), "건 (제조업등", len(mf), "+ 건설업", len(cs), ")")
print("통계표: 재해자수", inj.shape, "| 사망자수", dead.shape, "| 사업장수", firms.shape, "| 사망만인율", rate.shape)

## 1. 기술통계 요약 (M3-1)
사고사례(1번)를 업종구분별로 요약합니다. 제조업 `기인물`의 `분류 불가`는 M2에서 `기인물_표준`을 NaN으로 처리했습니다.

In [ ]:
g = sif.groupby("업종구분")
summary = pd.DataFrame({
    "건수": g.size(),
    "기인물 종류 수": g["기인물_표준"].nunique(),
    "기인물 결측(분류불가)": g["기인물_표준"].apply(lambda s: int(s.isna().sum())),
    "재해연도 최소": g["재해연도"].min(),
    "재해연도 최대": g["재해연도"].max(),
    "재해개요 평균 글자수": g["재해개요"].apply(lambda s: round(s.str.len().mean())),
})
display(summary)
summary.to_csv(A / "m3_기술통계요약.csv", encoding="utf-8-sig")

# 통계표(2~5번)는 숫자 컬럼만 요약
for name, d in [("2번 재해자수", inj), ("4번 사업장수", firms), ("5번 사망만인율", rate)]:
    num = d.select_dtypes("number")
    print(f"[{name}] 업종 {len(d)}개 × 규모 {num.shape[1]}개 | 결측 {int(num.isna().sum().sum())}칸 | 최소 {num.min().min():,.2f} | 중앙값 {num.stack().median():,.2f} | 최대 {num.max().max():,.2f}")

## 2. 분포 · 추이 (M3-2)
- 그래프 1: 사고사례의 재해연도별 건수 (연도는 `재해개요`에서 추출한 값)
- 그래프 2: 규모별 사고사망자수 합계의 연도별 추이 (3번 통계, 2004~2025)

In [ ]:
trend = sif.groupby(["재해연도", "업종구분"]).size().unstack(fill_value=0)
display(trend)
trend.to_csv(A / "m3_연도별사례수.csv", encoding="utf-8-sig")

ax = trend.plot(kind="bar", stacked=True, figsize=(9, 4))
ax.set_title("그래프 1. SIF 사고사례 연도별 건수")
ax.set_xlabel("재해연도"); ax.set_ylabel("건수")
plt.xticks(rotation=0); plt.tight_layout(); plt.show()
print("※ 사례는 2016~2023년에 집중돼 있고 나머지 연도는 매우 적다. 연도별 증감을 실제 사고 증감으로 해석하지 않는다.")

In [ ]:
years = [c for c in dead.columns if c.endswith("년")]
total = dead.set_index("구분")[years].sum()
total.index = [int(y[:-1]) for y in years]
display(total.to_frame("사고사망자수 합계").T)

ax = total.plot(marker="o", figsize=(9, 4))
ax.set_title("그래프 2. 연도별 사고사망자수 합계 (규모별 합산)")
ax.set_xlabel("연도"); ax.set_ylabel("사망자수")
ax.grid(alpha=.3); plt.tight_layout(); plt.show()
print(f"최고 {total.idxmax()}년 {int(total.max())}명 / 최저 {total.idxmin()}년 {int(total.min())}명 / 2004→2025 변화 {int(total.iloc[-1] - total.iloc[0]):+d}명")

## 3. 주요 범주 비교 (M3-3)
업종구분별로 사고 건수가 많은 `기인물` 상위 8개를 비교합니다. (제조업의 `분류 불가`는 제외)

In [ ]:
top = (sif.dropna(subset=["기인물_표준"]).groupby(["업종구분", "기인물_표준"]).size()
         .reset_index(name="건수").sort_values(["업종구분", "건수"], ascending=[True, False])
         .groupby("업종구분").head(8))
valid = sif.dropna(subset=["기인물_표준"]).groupby("업종구분").size()
top["비중(%)"] = (top["건수"] / top["업종구분"].map(valid) * 100).round(1)
display(top)
top.to_csv(A / "m3_기인물상위.csv", index=False, encoding="utf-8-sig")

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for ax, (kind, d) in zip(axes, top.groupby("업종구분")):
    d = d.sort_values("건수")
    ax.barh(d["기인물_표준"], d["건수"])
    ax.set_title(f"{kind} 기인물 상위 8")
    ax.set_xlabel("건수")
fig.suptitle("그래프 3. 업종구분별 주요 기인물")
plt.tight_layout(); plt.show()

# 집중도: 상위 8개가 전체(기인물 있는 사례)에서 차지하는 비중
conc = top.groupby("업종구분")["건수"].sum() / valid * 100
print("상위 8개 기인물 집중도(%):", conc.round(1).to_dict())

# 건설업은 재해종류도 비교
display(cs["재해종류"].value_counts().head(8).to_frame("건수"))

## 4. 변수 간 관계 (M3-4)
2번(재해자수)과 4번(사업장수)을 업종·규모로 맞춰 **규모별 사업장당 재해자수**를 구하고, 사업장수와 재해자수의 상관(Spearman)을 확인합니다.
4번 컬럼명은 `…사업장수`가 붙어 있어 규모명으로 맞춥니다.

## 필요 패키지
- 이 노트북은 **pandas, matplotlib** 를 사용합니다. 프로젝트 환경(uv)에 설치되어 있어야 합니다.
- `ModuleNotFoundError`가 나면 터미널에서 아래처럼 설치한 뒤 커널을 재시작하세요:
   - uv add pandas matplotlib

In [ ]:
sizes = [c for c in inj.columns if c not in ("대업종", "구분")]
firms2 = firms.rename(columns=lambda c: c.replace("사업장수", ""))
rate2 = rate.copy()
assert (inj["구분"].values == firms2["구분"].values).all(), "2번과 4번의 업종 순서가 다릅니다."

by_size = pd.DataFrame({
    "재해자수 합계": inj[sizes].sum(),
    "사업장수 합계": firms2[sizes].sum(),
})
by_size["사업장 1,000곳당 재해자수"] = (by_size["재해자수 합계"] / by_size["사업장수 합계"] * 1000).round(1)
display(by_size)
by_size.to_csv(A / "m3_규모별_사업장당재해자.csv", encoding="utf-8-sig")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
by_size["재해자수 합계"].plot(kind="bar", ax=axes[0]); axes[0].set_title("규모별 재해자수 합계")
by_size["사업장 1,000곳당 재해자수"].plot(kind="bar", ax=axes[1], color="tab:orange"); axes[1].set_title("규모별 사업장 1,000곳당 재해자수")
for ax in axes: ax.tick_params(axis="x", rotation=45)
fig.suptitle("그래프 4. 규모별 재해자수 vs 사업장당 재해자수")
plt.tight_layout(); plt.show()

# 업종×규모 칸 단위 상관 (사업장이 있는 칸만)
long = pd.DataFrame({
    "재해자수": inj[sizes].to_numpy().ravel(),
    "사업장수": firms2[sizes].to_numpy().ravel(),
}).dropna()
long = long[long["사업장수"] > 0]
rho = long["재해자수"].rank().corr(long["사업장수"].rank())
print(f"업종×규모 {len(long)}칸, 사업장수-재해자수 Spearman 상관계수 = {rho:.2f}")

In [ ]:
# 사망만인율: 업종별 중앙값(칸 단위, 결측 제외)과 극단값
med = rate.set_index("구분")[sizes].median(axis=1).dropna().sort_values(ascending=False)
top10 = med.head(10).sort_values()
ax = top10.plot(kind="barh", figsize=(9, 4.5))
ax.set_title("그래프 5. 사망만인율 중앙값 상위 10 업종 (규모 10구간 기준)")
ax.set_xlabel("사망만인율 (로그 눈금)"); ax.set_xscale("log")
plt.tight_layout(); plt.show()

extreme = rate.set_index("구분")[sizes].stack().sort_values(ascending=False).head(5)
display(extreme.to_frame("사망만인율"))
print("※ 소규모 모집단에서는 사망 1건이 비율을 크게 키우므로 극단값은 삭제하지 않고 따로 표기합니다 (M2 명세서).")

## 5. 핵심 인사이트 (M3-7)
아래 문장은 위 결과에서 계산한 값으로 만들어집니다.

In [ ]:
def pct(kind):
    return conc[kind]

mf_top = top[top["업종구분"] == "제조업등"].iloc[0]
cs_top = top[top["업종구분"] == "건설업"].iloc[0]
cs_names = ", ".join(top[top["업종구분"] == "건설업"]["기인물_표준"].head(4))
mf_names = ", ".join(top[top["업종구분"] == "제조업등"]["기인물_표준"].head(4))
best = by_size["사업장 1,000곳당 재해자수"].idxmax()

print(f"1. 업종별로 사고 유형이 갈린다. 제조업등은 {mf_names} 순, 건설업은 {cs_names} 등 고소작업 기인물이 상위다.")
print(f"2. 건설업은 소수 기인물에 집중(상위 8개 {pct('건설업'):.1f}%)하고 제조업등은 분산(상위 8개 {pct('제조업등'):.1f}%)된다.")
print(f"3. 사업장 1,000곳당 재해자수는 '{best}' 구간이 가장 높다 ({by_size.loc[best, '사업장 1,000곳당 재해자수']:.1f}명).")
print(f"4. 제조업 기인물 '분류 불가' {int(summary.loc['제조업등', '기인물 결측(분류불가)'])}건은 분석에서 결측으로 처리했다.")
print(f"5. 사망자수는 2004년 {int(total.iloc[0])}명에서 2025년 {int(total.iloc[-1])}명으로 변했다.")
print("→ RAG 설계 시사점: 업종구분·기인물을 메타데이터 필터로 쓰면 검색 범위를 좁힐 수 있다 (M4/M7).")